In [84]:
link = r"C:\Users\jeand\Documents\horairesMontigny.txt"
#link = r"C:\Users\jeand\Documents\horairesRER_E_matin.txt"
with open(link, encoding="utf-8") as fichier:
    contenu = fichier.read()

#print(contenu)

In [85]:
import json
import pandas as pd

data = json.loads(contenu)

visites = (
    data["Siri"]["ServiceDelivery"]["StopMonitoringDelivery"][0]["MonitoredStopVisit"]
)


In [86]:
firsttrain = visites[0]["MonitoredVehicleJourney"]
firsttrain

{'LineRef': {'value': 'STIF:Line::C01727:'},
 'OperatorRef': {},
 'FramedVehicleJourneyRef': {'DataFrameRef': {'value': 'any'},
  'DatedVehicleJourneyRef': 'SNCF_MAGENTA_PRD:VehicleJourney::c05b2a33-6212-4b93-9756-aceb68a9a9de:LOC'},
 'DirectionName': [],
 'DestinationRef': {'value': 'STIF:StopArea:SP:43166:'},
 'DestinationName': [{'value': 'Montigny - Beauchamp'}],
 'VehicleJourneyName': [{'value': '148106-148107'}],
 'JourneyNote': [{'value': 'GOTA'}],
 'MonitoredCall': {'StopPointName': [{'value': 'Montigny - Beauchamp'}],
  'VehicleAtStop': False,
  'DestinationDisplay': [{'value': 'Montigny - Beauchamp'}],
  'ArrivalStopAssignment': {'ExpectedQuayRef': {'value': 'STIF:StopPoint:Q:471725:'}},
  'ExpectedArrivalTime': '2026-10-23T11:35:02.000Z',
  'DeparturePlatformName': {'value': 'M'},
  'DepartureStatus': 'onTime',
  'Order': 33,
  'AimedArrivalTime': '2026-10-23T11:34:30.000Z',
  'ArrivalPlatformName': {'value': 'M'},
  'ArrivalStatus': 'onTime'},
 'TrainNumbers': {'TrainNumber

In [87]:
# first train
journey = json.dumps(visites[0]["MonitoredVehicleJourney"], ensure_ascii=False, indent=2)
print(journey)

{
  "LineRef": {
    "value": "STIF:Line::C01727:"
  },
  "OperatorRef": {},
  "FramedVehicleJourneyRef": {
    "DataFrameRef": {
      "value": "any"
    },
    "DatedVehicleJourneyRef": "SNCF_MAGENTA_PRD:VehicleJourney::c05b2a33-6212-4b93-9756-aceb68a9a9de:LOC"
  },
  "DirectionName": [],
  "DestinationRef": {
    "value": "STIF:StopArea:SP:43166:"
  },
  "DestinationName": [
    {
      "value": "Montigny - Beauchamp"
    }
  ],
  "VehicleJourneyName": [
    {
      "value": "148106-148107"
    }
  ],
  "JourneyNote": [
    {
      "value": "GOTA"
    }
  ],
  "MonitoredCall": {
    "StopPointName": [
      {
        "value": "Montigny - Beauchamp"
      }
    ],
    "VehicleAtStop": false,
    "DestinationDisplay": [
      {
        "value": "Montigny - Beauchamp"
      }
    ],
    "ArrivalStopAssignment": {
      "ExpectedQuayRef": {
        "value": "STIF:StopPoint:Q:471725:"
      }
    },
    "ExpectedArrivalTime": "2026-10-23T11:35:02.000Z",
    "DeparturePlatformName": {
   

In [88]:
df_train = pd.json_normalize(visites[0]["MonitoredVehicleJourney"], sep="_")
df_train[
    [
        #"LineRef_value",
        #"FramedVehicleJourneyRef_DataFrameRef_value",
        #"FramedVehicleJourneyRef_DatedVehicleJourneyRef",
        #"DirectionName",
        "DestinationRef_value",
        "DestinationName",
        "VehicleJourneyName",
        "JourneyNote",
        "VehicleFeatureRef",
        "DirectionRef_value",
        "MonitoredCall_StopPointName",
        "MonitoredCall_ExpectedArrivalTime",
        #"MonitoredCall_ExpectedDepartureTime",
    ]
]

,DestinationRef_value,DestinationName,VehicleJourneyName,JourneyNote,VehicleFeatureRef,DirectionRef_value,MonitoredCall_StopPointName,MonitoredCall_ExpectedArrivalTime
0,STIF:StopArea:SP:43166:,[{'value': 'Montigny - Beauchamp'}],[{'value': '148106-148107'}],[{'value': 'GOTA'}],[longTrain],Retour,[{'value': 'Montigny - Beauchamp'}],2026-10-23T11:35:02.000Z


In [89]:
# taille premier train
feature = json.dumps(firsttrain["VehicleFeatureRef"], indent=4)
size = json.loads(feature)
print(size[0])

longTrain


In [90]:
firsttrain.keys()

dict_keys(['LineRef', 'OperatorRef', 'FramedVehicleJourneyRef', 'DirectionName', 'DestinationRef', 'DestinationName', 'VehicleJourneyName', 'JourneyNote', 'MonitoredCall', 'TrainNumbers', 'VehicleFeatureRef', 'DirectionRef'])

In [91]:
print(df_train['VehicleFeatureRef'][0][0])              #['VehicleFeatureRef'].get('value'))

longTrain


In [92]:
# get all trains informations
lignes = []
for visite in visites:
    trajet = visite["MonitoredVehicleJourney"]
    appel = trajet["MonitoredCall"]

    lignes.append({
        "train": trajet.get("VehicleJourneyName", [{}])[0].get("value"),
        "direction": trajet.get("DirectionRef", {}).get("value"),
        "destination": trajet.get("DestinationName", [{}])[0].get("value"),
        #"gare": appel.get("StopPointName", [{}])[0].get("value"),
        "arrivee_prevue": appel.get("ExpectedArrivalTime"),
        "depart_prevu": appel.get("ExpectedDepartureTime"),
        "voie": appel.get("DeparturePlatformName", {}).get("value"),
        "statut_depart": appel.get("DepartureStatus"),
        "ordre": appel.get("Order"),
        "size": trajet.get("VehicleFeatureRef", {})[0],
    })

df = pd.DataFrame(lignes)

# Convertir les horaires en dates pandas
for column in ["arrivee_prevue", "depart_prevu"]:
    df[column] = pd.to_datetime(df[column]).dt.strftime("%H:%M")
#df["arrivee_prevue"] = pd.to_datetime(df["arrivee_prevue"])
#df["depart_prevu"] = pd.to_datetime(df["depart_prevu"])

df

,train,direction,destination,arrivee_prevue,depart_prevu,voie,statut_depart,ordre,size
0,148106-148107,Retour,Montigny - Beauchamp,11:35,NaN,M,onTime,33,longTrain
1,123473,Aller,Pontoise,12:09,12:10,1,onTime,13,shortTrain
2,123508,Retour,Paris Gare du Nord,12:18,12:18,2,onTime,19,longTrain
3,148512-148513,Retour,Pontoise,12:18,12:19,1,onTime,72,longTrain
4,147126-147127,Aller,Pont de Rungis Aéroport d'Orly,NaN,12:25,M,onTime,31,longTrain
5,123475,Aller,Pontoise,12:39,12:40,1,onTime,24,shortTrain
6,147528-147529,Aller,Massy-Palaiseau,12:40,12:41,2,onTime,43,longTrain
7,123510,Retour,Paris Gare du Nord,12:48,12:48,2,onTime,19,longTrain
8,147130-147131,Aller,Pont de Rungis Aéroport d'Orly,NaN,12:55,M,cancelled,31,longTrain
9,123477,Aller,Pontoise,13:09,13:10,1,onTime,24,shortTrain
